In [1]:
import torch
import numpy as np
from ultralytics import YOLO

# Load YOLO model
model = YOLO("yolov5nu.pt")
state_dict = model.model.state_dict()

# Flattened array and layer pointer dict
flat_weights = []
layer_ptrs = {}
current_ptr = 0

# Helper: Only consider submodules that have weights/biases
for name, param in state_dict.items():
    w = param.cpu().numpy().flatten()  # flatten to 1D
    start_idx = current_ptr
    end_idx = current_ptr + w.size  # exclusive
    flat_weights.extend(w)
    # Save start and end indices in layer_ptrs
    layer_ptrs[name] = (start_idx, end_idx)
    current_ptr = end_idx

# Convert to a single NumPy array
flat_weights = np.array(flat_weights, dtype=np.float32)

print(f"Total weights: {flat_weights.size}")
print(f"Number of layers: {len(layer_ptrs)}")
print(f"Example layer: {list(layer_ptrs.items())[0]}")

# Optional: save
# np.savez("yolo_flat_weights.npz", flat_weights=flat_weights, layer_ptrs=layer_ptrs)

/home/geeth/miniconda3/envs/yolov13/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FlashAttention is not available on this device. Using scaled_dot_product_attention instead.
Total weights: 2666117
Number of layers: 427
Example layer: ('model.0.conv.weight', (0, 1728))


In [ ]:
import numpy as np

num_layers = len(layer_ptrs)  # number of layers with weights
fields_per_layer = 12

# Initialize a flat array with zeros
model_struct_flat = np.zeros(num_layers * fields_per_layer, dtype=np.int64)

print(f"Flat model_struct initialized with {model_struct_flat.size} elements")

# Helper function to fill a layer
def set_layer(flat_array, layer_idx, layer_id, ptr_in=0, num_in=0, ptr_out=0, num_out=0,
              ptr_input_features=0, ptr_module=0, module_end=0, ptr_args=0, num_args=0,
              ptr_weights=0, num_weights=0):
    start = layer_idx * fields_per_layer
    flat_array[start:start+fields_per_layer] = [
        layer_id,      # 0
        ptr_in,        # 1
        num_in,        # 2
        ptr_out,       # 3
        num_out,       # 4
        ptr_input_features,  # 5
        ptr_module,    # 6
        module_end,    # 7
        ptr_args,      # 8
        num_args,      # 9
        ptr_weights,   # 10
        num_weights    # 11
    ]

# Populate all layers
for layer_idx, (layer_name, (start, end)) in enumerate(layer_ptrs.items()):
    num_weights = end - start
    ptr_weights = start
    # Fill the layer using set_layer
    set_layer(model_struct_flat, layer_idx, layer_idx, ptr_weights=ptr_weights, num_weights=num_weights)

# Verify first few layers
for i in range(min(5, len(layer_ptrs))):
    start_slot = i * fields_per_layer
    print(f"Layer {i} -> layer_id: {model_struct_flat[start_slot]}, ptr_weights: {model_struct_flat[start_slot+10]}, num_weights: {model_struct_flat[start_slot+11]}")



Flat model_struct initialized with 5124 elements
Layer 0 -> layer_id: 0, ptr_weights: 0, num_weights: 1728
Layer 1 -> layer_id: 1, ptr_weights: 1728, num_weights: 16
Layer 2 -> layer_id: 2, ptr_weights: 1744, num_weights: 16
Layer 3 -> layer_id: 3, ptr_weights: 1760, num_weights: 16
Layer 4 -> layer_id: 4, ptr_weights: 1776, num_weights: 16
[   0    0    0    0    0    0    0    0    0    0    0 1728    1    0    0    0    0    0    0    0    0    0 1728   16    2    0    0    0    0    0    0    0    0    0 1744   16    3    0    0    0    0    0    0    0    0    0 1760   16    4    0    0    0    0    0    0    0    0    0 1776   16    5    0    0
    0    0    0    0    0    0    0 1792    1    6    0    0    0    0    0    0    0    0    0 1793 4608    7    0    0    0    0    0    0    0    0    0 6401   32    8    0    0    0]


In [ ]:
weights = flat_weights
ptr_map = layer_ptrs
model_struct = model_struct_flat
# [layer_id, ptr_in, num_in, ptr_out, num_out, ptr_input_features,
#   ptr_module, module_end, ptr_args, num_args, ptr_weights, num_weights ]
print(ptr_map)
print(model_struct_flat[:100])

{'model.0.conv.weight': (0, 1728), 'model.0.bn.weight': (1728, 1744), 'model.0.bn.bias': (1744, 1760), 'model.0.bn.running_mean': (1760, 1776), 'model.0.bn.running_var': (1776, 1792), 'model.0.bn.num_batches_tracked': (1792, 1793), 'model.1.conv.weight': (1793, 6401), 'model.1.bn.weight': (6401, 6433), 'model.1.bn.bias': (6433, 6465), 'model.1.bn.running_mean': (6465, 6497), 'model.1.bn.running_var': (6497, 6529), 'model.1.bn.num_batches_tracked': (6529, 6530), 'model.2.cv1.conv.weight': (6530, 7042), 'model.2.cv1.bn.weight': (7042, 7058), 'model.2.cv1.bn.bias': (7058, 7074), 'model.2.cv1.bn.running_mean': (7074, 7090), 'model.2.cv1.bn.running_var': (7090, 7106), 'model.2.cv1.bn.num_batches_tracked': (7106, 7107), 'model.2.cv2.conv.weight': (7107, 7619), 'model.2.cv2.bn.weight': (7619, 7635), 'model.2.cv2.bn.bias': (7635, 7651), 'model.2.cv2.bn.running_mean': (7651, 7667), 'model.2.cv2.bn.running_var': (7667, 7683), 'model.2.cv2.bn.num_batches_tracked': (7683, 7684), 'model.2.cv3.conv.